# Data Cleaning Notebook for NLP Features (V2)

This notebook provides a clear, step-by-step workflow to find and manually fix rows with missing (`NaN`) feature values in your generated CSV files. 

**Follow the steps in order from top to bottom.**

### Setup: Defining the Toolkit Functions

Run the cell below **once** to define all the necessary helper functions.

In [1]:
import pandas as pd
import json
import os
from IPython.display import display, clear_output

# --- Configuration ---
FEATURE_COLUMNS = [
    'News Relevance', 'Sentiment', 'Price Impact Potential', 
    'Trend Direction', 'Earnings Impact', 'Investor Confidence', 
    'Risk Profile Change'
]

# --- Function 1: Load and Analyze ---
def load_and_analyze_file(file_path: str):
    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        return None
    print(f"Loading file: {os.path.basename(file_path)}...")
    df = pd.read_csv(file_path)
    nan_rows_mask = df[FEATURE_COLUMNS].isnull().any(axis=1)
    nan_count = len(df[nan_rows_mask])
    print(f"File loaded successfully. Total rows: {len(df)}")
    if nan_count > 0:
        print(f"Found {nan_count} rows with at least one missing NLP feature value.")
    else:
        print("No missing values found in this file. It looks clean!")
    return df

# --- Function 2: Get the Next Task ---
def get_next_fix_task(df: pd.DataFrame):
    nan_rows_mask = df[FEATURE_COLUMNS].isnull().any(axis=1)
    nan_rows = df[nan_rows_mask]
    if nan_rows.empty:
        print("✅ Congratulations! No more missing values to fix in this DataFrame.")
        return None, None
    row_index = nan_rows.index[0]
    prompt = df.loc[row_index, 'Prompt']
    date = df.loc[row_index, 'Date']
    clear_output(wait=True)
    print("="*50)
    print(f"Next task: Fix row with Index: {row_index} (Date: {date})")
    print("="*50)
    print("\n[COPY THE FOLLOWING PROMPT AND PASTE INTO CHATGPT]\n")
    print(prompt)
    print("\n" + "-"*50)
    return row_index, prompt

# --- Function 3: Update a Specific Row ---
def update_row_from_json(df: pd.DataFrame, row_index: int, json_response_string: str):
    if '"paste": "your JSON here"' in json_response_string:
        print("[ERROR] You have not pasted the JSON response yet. Please go to the 'Step 3' cell, paste the JSON, and run it before running this cell.")
        return df
    try:
        output_data = json.loads(json_response_string)
        print(f"\nUpdating row {row_index} with new values...")
        for col_name in FEATURE_COLUMNS:
            json_key = col_name.lower().replace(' ', '_')
            if json_key in output_data:
                value = int(output_data.get(json_key, 0))
                df.loc[row_index, col_name] = value
                print(f"  - Set '{col_name}' to {value}")
            else:
                print(f"  - WARNING: Key '{json_key}' not found in JSON response. Skipping column '{col_name}'.")
        print(f"\nSuccessfully updated row {row_index}.")
        return df
    except json.JSONDecodeError:
        print("\n[ERROR] Invalid JSON provided. Please make sure you copy and paste only the JSON code block. The DataFrame was not changed.")
        return df
    except Exception as e:
        print(f"\n[ERROR] An unexpected error occurred: {e}. The DataFrame was not changed.")
        return df

### Step 1: Configure and Load Your File

Set the `file_to_fix` variable to the full path of the CSV file you want to clean. Then, run the cell to load the data and see how many rows need fixing.

In [163]:
# Set the full path to the file you want to fix
file_to_fix = 'data_gpt/NFLX_2020-07-01_2025-05-31/NFLX_2020-07-01_2025-05-31_gpt.csv'

# Load the file and get a report on missing values
df = load_and_analyze_file(file_to_fix)

Loading file: NFLX_2020-07-01_2025-05-31_gpt.csv...
File loaded successfully. Total rows: 1233
No missing values found in this file. It looks clean!


### Step 2: Get the Next Task

Run the cell below. It will find the **next** row that needs fixing and print its prompt. Then, move to Step 3.

In [160]:
# This cell finds the next row to fix.
if 'df' in locals() and df is not None:
    # This stores the index and prompt in memory for the next step
    index_to_fix, prompt_to_run = get_next_fix_task(df)
else:
    print("Please run Step 1 to load a DataFrame first.")

✅ Congratulations! No more missing values to fix in this DataFrame.


### Step 3: Paste the ChatGPT Response

After getting the response from ChatGPT, paste the **complete JSON object** into the `chatgpt_response` variable in the cell below and run it to store the response.

In [158]:
# Paste the JSON response from ChatGPT inside the triple quotes.
chatgpt_response = """
{
	"news_relevance": "2",
	"sentiment": "1",
	"price_impact_potential": "2",
	"trend_direction": "1",
	"earnings_impact": "2",
	"investor_confidence": "2",
	"risk_profile_change": "0"
}
"""

print("Response stored successfully. Now run Step 4 to apply it.")

Response stored successfully. Now run Step 4 to apply it.


### Step 4: Update the Row and Verify

Run this cell to apply the changes to the DataFrame. It will use the task from **Step 2** and the response you pasted in **Step 3**.

In [159]:
# This cell performs the update.
if 'df' in locals() and 'index_to_fix' in locals() and index_to_fix is not None:
    # Update the row using the stored index and response
    df = update_row_from_json(df, index_to_fix, chatgpt_response)
    
    # Display the row you just fixed to verify the changes
    print("\nVerification of the updated row:")
    display(df.loc[[index_to_fix]])

    # Clear the variables to prevent accidentally re-updating the same row
    del index_to_fix
    del prompt_to_run
else:
    print("Please run Step 2 to get a new task first.")


Updating row 1212 with new values...
  - Set 'News Relevance' to 2
  - Set 'Sentiment' to 1
  - Set 'Price Impact Potential' to 2
  - Set 'Trend Direction' to 1
  - Set 'Earnings Impact' to 2
  - Set 'Investor Confidence' to 2
  - Set 'Risk Profile Change' to 0

Successfully updated row 1212.

Verification of the updated row:


,Date,Adj Close Price,Returns,Bin Label,News Relevance,Sentiment,Price Impact Potential,Trend Direction,Earnings Impact,Investor Confidence,Risk Profile Change,Prompt
1212,2025-04-30,394.535706,0.003096,U1,2.0,1.0,2.0,1.0,2.0,2.0,0.0,Human: \n [SYSTEM PROMPT]\n You are a se...


In [49]:
# The index of the row you want to fix
row_index_to_fix = 184

# The dictionary containing the correct values for each column
correct_values = {
    "News Relevance": 1,
    "Sentiment": -1,
    "Price Impact Potential": -1,
    "Trend Direction": 0,
    "Earnings Impact": 0,
    "Investor Confidence": -1,
    "Risk Profile Change": -1
}

# --- Update Logic ---

if 'df' in locals() and df is not None:
    try:
        print(f"Attempting to overwrite row {row_index_to_fix} with new data...")
        
        # Loop through the dictionary and update each column for the specified row
        for column, value in correct_values.items():
            df.loc[row_index_to_fix, column] = value
            
        print("Update successful!")
        
        # Display the updated row to verify the changes
        print("\nVerification of the corrected row:")
        display(df.loc[[row_index_to_fix]])
        
    except KeyError:
        print(f"[ERROR] Could not find row with index {row_index_to_fix}. Please make sure the DataFrame is loaded and the index exists.")
    except Exception as e:
        print(f"[ERROR] An unexpected error occurred: {e}")
else:
    print("[ERROR] DataFrame 'df' not found. Please load your file first using 'Step 1' from the notebook.")

Attempting to overwrite row 184 with new data...
Update successful!

Verification of the corrected row:


,Date,Adj Close Price,Returns,Bin Label,News Relevance,Sentiment,Price Impact Potential,Trend Direction,Earnings Impact,Investor Confidence,Risk Profile Change,Prompt
184,2021-03-26,152.601501,0.001894,U1,1.0,-1.0,-1.0,0.0,0.0,-1.0,-1.0,Human: \n [SYSTEM PROMPT]\n You are a se...


### Loop back to Step 2 to fix the next row.
---

### Step 5: Save Your Cleaned Data

**Only run this cell when you are finished!** (When Step 2 tells you there are no more missing values).

This will save your updated DataFrame, overwriting the original file with the clean version.

In [161]:
if 'df' in locals() and df is not None:
    # Save the cleaned DataFrame back to its original file
    df.to_csv(file_to_fix, index=False)
    print(f"✅ Successfully saved cleaned data to: {file_to_fix}")
else:
    print("No DataFrame loaded. Please run Step 1 first.")

✅ Successfully saved cleaned data to: data_gpt/MSFT_2020-07-01_2025-05-31/MSFT_2020-07-01_2025-05-31_gpt.csv
